# 01 · Descarga de datos (S4)

Baja, para las 9 manchas de la muestra, los cuatro productos HMI a **720 s**:

| producto | serie JSOC | forma |
|---|---|---|
| continuo | `hmi.Ic_720s` | cutout con seguimiento (`im_patch`), malla HPC |
| magnetograma | `hmi.M_720s` | cutout con seguimiento, malla HPC |
| dopplergrama | `hmi.V_720s` | cutout con seguimiento, malla HPC |
| vector SHARP | `hmi.sharp_cea_720s` | HARP completo, malla CEA (Br, Bp, Bt, errores, bitmap, conf_disambig) |

**Ventanas de tiempo**
- *Löptien* (validación): el día completo de la fecha del barrido de Hinode. Es la misma ventana que `FIT_WINDOWS` del proyecto de oscilaciones, porque no hay registro de la hora exacta del barrido.
- *Catálogo propio* (predicción): las 48 h en que la mancha está más centrada en el disco, calculadas con las posiciones SRS de NOAA.

**Detalles heredados del proyecto `sun_spots_pulsation`**
- La caja del cutout la da `locate_ar_window`: es la caja HEK del grupo de manchas de NOAA SWPC.
- 11536 usa la caja más grande (`BOX_OVERRIDE`).
- 11117 baja las series de 45 s submuestreadas a 720 s, porque las de 720 s dan error 500 en JSOC.
- Las descargas se reanudan: correr de nuevo el notebook solo baja lo que falta.


In [ ]:
import pathlib, sys, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from src import config, download
print('JSOC email:', config.NOTIFY_EMAIL)

## Parámetros

- `REUSE_PREVIOUS`: copia los frames que el proyecto de oscilaciones ya bajó para las manchas de Löptien (misma caja, mismas series). Así no se vuelven a pedir unos 4 GB a JSOC.
- `DRY_RUN_HOURS`: si no es `None`, recorta todas las ventanas a sus primeras N horas. Sirve para probar el flujo completo bajando poco.
- `ONLY`: lista de `raw_id` para bajar solo esas carpetas (por ejemplo `['C12670']`); `None` baja todas.

In [ ]:
REUSE_PREVIOUS = True
DRY_RUN_HOURS = None   # p. ej. 1 para una prueba rápida
ONLY = None

## 1. Ventanas de tiempo

Para el catálogo propio, `most_centered_window` hace estos pasos:
1. Ajusta una recta longitud(t) a las posiciones SRS diarias (≈13.2°/día).
2. Busca el instante del paso por el meridiano central (lon = 0).
3. Toma ±24 h alrededor de ese instante.

A latitud fija, μ es máximo en lon = 0 y simétrico alrededor, así que esa ventana es la de μ medio máximo. `mu_mean` y `mu_edge` son aproximados (sin B0).

In [ ]:
rows = []
for s in config.LOPTIEN_SPOTS:
    rows.append(dict(id=s['id'], noaa=s['noaa'], raw_id=s['raw_id'], harpnum=s['harpnum'], kind='loptien',
                     t_start=s['t_start'], t_end=s['t_end']))
for s in config.CATALOG_SPOTS:
    w = download.most_centered_window(config.CATALOG_DIR / f"srs_{s['noaa']}.csv", hours=config.WINDOW_HOURS)
    rows.append(dict(id=s['id'], noaa=s['noaa'], raw_id=s['raw_id'], harpnum=s['harpnum'], kind='catalog',
                     t_start=w['t_start'], t_end=w['t_end'], t_cm=w['t_cm'],
                     mu_mean=w['mu_mean'], mu_edge=w['mu_edge'], rate_deg_day=w['rate_deg_per_day']))
windows = pd.DataFrame(rows)
for c in ('t_start', 't_end', 't_cm'):
    windows[c] = pd.to_datetime(windows[c])
if DRY_RUN_HOURS is not None:
    windows['t_end'] = windows['t_start'] + pd.Timedelta(hours=DRY_RUN_HOURS) - pd.Timedelta(seconds=1)
windows.to_csv(config.RAW_DIR / 'windows.csv', index=False)
windows

Comprobación del HARPNUM. Un HARP puede agrupar varias regiones: 1879 = 11529–11536 y 6699 = 12573–12577. Por eso la mancha se ubica con la caja SWPC y no con el centro del HARP.

In [ ]:
for s in config.all_spots():
    t = pd.Timestamp(windows.set_index('id').loc[s['id'], 't_start']) + pd.Timedelta(hours=1)
    hit = download.find_harpnum(s['noaa'], t)
    ok = s['harpnum'] in hit['HARPNUM'].tolist()
    print(f"{s['id']:16s} HARP config {s['harpnum']:5d}  JSOC {hit['HARPNUM'].tolist()}  NOAA_ARS {hit['NOAA_ARS'].tolist()}  {'ok' if ok else 'REVISAR'}")

## 2. Una descarga por carpeta `raw_id`

Las dos fechas de 11117 comparten la carpeta `L11117`: su ventana de descarga es la unión, 27 y 28 de octubre.

In [ ]:
groups = (windows.groupby('raw_id')
          .agg(noaa=('noaa', 'first'), harpnum=('harpnum', 'first'), kind=('kind', 'first'),
               t_start=('t_start', 'min'), t_end=('t_end', 'max')))
if ONLY is not None:
    groups = groups.loc[ONLY]
groups

## 3. Reutilizar frames del proyecto de oscilaciones (solo Löptien)

In [ ]:
if REUSE_PREVIOUS:
    for raw_id, g in groups.iterrows():
        prev = config.PREVIOUS_RAW_DIRS.get(raw_id)
        if prev is None:
            continue
        src = config.PREVIOUS_PROJECT_RAW / prev / 'region_01'
        for series in config.series_for(g['noaa']):
            meta = download.SERIES_META[series]
            n = download.reuse_previous_frames(src, config.RAW_DIR / raw_id / meta['label'],
                                               meta['glob'], g['t_start'], g['t_end'])
            print(f'{raw_id} {series}: {n} archivos copiados de {prev}')

## 4. Cajas HEK y descarga de cutouts

La caja de Löptien se calcula con el mismo ancla y los mismos `n_days = 4` del proyecto original, para que coincida con los frames reutilizados. En el catálogo propio, el ancla es el primer día de la ventana.

`download_cutouts` devuelve uno de cuatro estados:
- `skipped`: ya estaba en disco.
- `ok`
- `partial`: faltan archivos tras el reintento. Hay que volver a correr la celda.
- `failed`

In [ ]:
status = []
boxes = {}
for raw_id, g in groups.iterrows():
    noaa = g['noaa']
    if g['kind'] == 'loptien':
        anchor, n_days = config.PREVIOUS_RAW_DIRS[raw_id].split('_')[-1], 4
    else:
        anchor = f"{g['t_start']:%Y-%m-%d}"
        n_days = (g['t_end'].normalize() - g['t_start'].normalize()).days + 1
    bl, tr, _, _ = download.locate_ar_window(noaa, anchor, n_days=n_days, **config.BOX_OVERRIDE.get(noaa, {}))
    boxes[raw_id] = (bl, tr)
    for series in config.series_for(noaa):
        meta = download.SERIES_META[series]
        r = download.download_cutouts(bl, tr, config.RAW_DIR / raw_id / meta['label'],
                                      str(g['t_start']), str(g['t_end']), config.NOTIFY_EMAIL,
                                      series, sample=config.sample_for(noaa))
        status.append(dict(raw_id=raw_id, product=series, **{k: v for k, v in r.items() if k != 'failed_urls'}))

## 5. Vector SHARP (`hmi.sharp_cea_720s`)

Se exporta con drms (`protocol='fits'`, así cada segmento trae todas sus keywords y su WCS CEA). JSOC permite **una** exportación pendiente por correo; `download_sharp` espera si hay otra en cola. El HARP 1879 (11536) es enorme, ~21 MB por registro con los 8 segmentos; en 02 se recorta a la zona del cutout.

In [ ]:
for raw_id, g in groups.iterrows():
    r = download.download_sharp(g['harpnum'], g['t_start'], g['t_end'], config.RAW_DIR / raw_id / 'sharp',
                                config.NOTIFY_EMAIL, series=config.SHARP_SERIES, segments=config.SHARP_SEGMENTS)
    status.append(dict(raw_id=raw_id, product=config.SHARP_SERIES, **r))

## 6. Estado final

Número de frames por producto en cada carpeta. Una ventana de 24 h a 720 s tiene 120 slots y una de 48 h tiene 240. Los faltantes que reporta JSOC se ven aquí y luego en 02, que solo usa los tiempos donde están los cuatro productos.

In [ ]:
status_df = pd.DataFrame(status)
display(status_df)

from src.download import PRODUCT_GLOBS
count = {}
for raw_id in groups.index:
    d = config.RAW_DIR / raw_id
    count[raw_id] = {p: len(list((d / p).glob(gl))) for p, gl in PRODUCT_GLOBS.items()}
    count[raw_id]['sharp_Br'] = len(list((d / 'sharp').glob('*.Br.fits')))
pd.DataFrame(count).T